In [19]:
# CELL 1: SETUP + ROBUST DATA LOADING

import os
import time
import zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from sklearn.metrics import roc_auc_score

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Using device:", device)

zip_path = '/content/Sign Language MNIST.zip'
extract_path = '/content/sign_language_mnist'

# Locate existing CSV files first
train_csv = None
test_csv = None

for root, dirs, files in os.walk('/content'):
    for f in files:
        if f.lower() == 'sign_mnist_train.csv':
            train_csv = os.path.join(root, f)
        elif f.lower() == 'sign_mnist_test.csv':
            test_csv = os.path.join(root, f)

# Extract only if CSV files are not already available
if train_csv is None or test_csv is None:

    if not os.path.exists(zip_path):
        raise FileNotFoundError(
            "ZIP file missing. Upload the dataset again."
        )

    # Validate ZIP archive
    if not zipfile.is_zipfile(zip_path):
        raise ValueError(
            "The uploaded file is NOT a valid ZIP archive.\n"
            "Delete the current ZIP and upload a fresh dataset ZIP."
        )

    os.makedirs(extract_path, exist_ok=True)

    with zipfile.ZipFile(zip_path, 'r') as z:
        bad_file = z.testzip()

        if bad_file is not None:
            raise ValueError(f"Corrupted ZIP entry: {bad_file}")

        z.extractall(extract_path)

    # Search extracted files
    train_csv = None
    test_csv = None

    for root, dirs, files in os.walk(extract_path):
        for f in files:
            if f.lower() == 'sign_mnist_train.csv':
                train_csv = os.path.join(root, f)
            elif f.lower() == 'sign_mnist_test.csv':
                test_csv = os.path.join(root, f)

if train_csv is None or test_csv is None:
    raise FileNotFoundError(
        "Training or test CSV could not be found.\n"
        "Check that the ZIP contains sign_mnist_train.csv "
        "and sign_mnist_test.csv."
    )

print("Train file:", train_csv)
print("Test file:", test_csv)

# Load data
train_df = pd.read_csv(train_csv)
test_df = pd.read_csv(test_csv)

y_train_raw = train_df['label'].values
X_train_raw = train_df.iloc[:, 1:].values.astype(np.float32)

y_test_raw = test_df['label'].values
X_test_raw = test_df.iloc[:, 1:].values.astype(np.float32)

# Map labels to continuous indices
unique_labels = np.unique(y_train_raw)
label_map = {label: idx for idx, label in enumerate(unique_labels)}

y_train = np.array([label_map[x] for x in y_train_raw])
y_test = np.array([label_map[x] for x in y_test_raw])

num_classes = len(unique_labels)

# Image normalization
def preprocess_images(X):
    imgs = X.reshape(-1, 1, 28, 28)
    mean = imgs.mean(axis=(2, 3), keepdims=True)
    std = imgs.std(axis=(2, 3), keepdims=True) + 1e-5
    return (imgs - mean) / std

X_train = torch.tensor(
    preprocess_images(X_train_raw),
    dtype=torch.float32,
    device=device
)

X_test = torch.tensor(
    preprocess_images(X_test_raw),
    dtype=torch.float32,
    device=device
)

y_train_t = torch.tensor(y_train, dtype=torch.long, device=device)
y_test_t = torch.tensor(y_test, dtype=torch.long, device=device)

print("Train:", X_train.shape)
print("Test:", X_test.shape)
print("Classes:", num_classes)
print("Data loaded successfully!")

Using device: cuda
Train file: /content/sign_language_mnist/sign_mnist_train/sign_mnist_train.csv
Test file: /content/sign_language_mnist/sign_mnist_test/sign_mnist_test.csv
Train: torch.Size([27455, 1, 28, 28])
Test: torch.Size([7172, 1, 28, 28])
Classes: 24
Data loaded successfully!


In [20]:
class Conv2DFPEnsembleBlock:
    def __init__(
        self, in_channels, out_channels, kernel_size,
        num_classes, R=5, lam=10.0,
        stride=1, padding=1, device='cpu'
    ):
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.num_classes = num_classes
        self.R = R
        self.lam = lam
        self.stride = stride
        self.padding = padding
        self.device = device

        self.patch_dim = in_channels * kernel_size * kernel_size + 1
        self.gammas = [1.0 / R] * R

        self.Q_list = [
            torch.randn(self.patch_dim, out_channels, device=device)
            / (self.patch_dim ** 0.5)
            for _ in range(R)
        ]

        self.U_list = [
            torch.randn(num_classes, out_channels, device=device)
            / (num_classes ** 0.5)
            for _ in range(R)
        ]

        self.W = None

    def _extract_patches(self, X, spatial_step=1):
        N, C, H, W = X.shape

        unfold = F.unfold(
            X,
            kernel_size=self.kernel_size,
            padding=self.padding,
            stride=self.stride
        )

        H_out = (H + 2 * self.padding - self.kernel_size) // self.stride + 1
        W_out = (W + 2 * self.padding - self.kernel_size) // self.stride + 1

        patches = unfold.transpose(1, 2)

        if spatial_step > 1:
            patches = patches.reshape(N, H_out, W_out, -1)
            patches = patches[:, ::spatial_step, ::spatial_step, :]
            patches = patches.reshape(N, -1, C * self.kernel_size * self.kernel_size)

        ones = torch.ones(
            (patches.shape[0], patches.shape[1], 1),
            device=self.device
        )

        patches = torch.cat([patches, ones], dim=-1)

        return patches, H_out, W_out

    def fit(self, X, Y_onehot, batch_size=512, spatial_subsample=2):
        t0 = time.time()
        t_target = 0.0
        N = X.shape[0]

        ATA = torch.zeros(
            self.patch_dim, self.patch_dim,
            device=self.device
        )

        ATZ = torch.zeros(
            self.patch_dim, self.out_channels,
            device=self.device
        )

        for start in range(0, N, batch_size):
            end = min(start + batch_size, N)

            xb = X[start:end]
            yb = Y_onehot[start:end]

            patches, _, _ = self._extract_patches(
                xb, spatial_step=spatial_subsample
            )

            B, L, D = patches.shape
            A_flat = patches.reshape(B * L, D)

            Y_flat = (
                yb.unsqueeze(1)
                .expand(B, L, self.num_classes)
                .reshape(B * L, self.num_classes)
            )

            tt0 = time.time()

            Z_ens = torch.zeros(
                B * L, self.out_channels,
                device=self.device
            )

            for r in range(self.R):
                Z_r = (
                    torch.sign(A_flat @ self.Q_list[r])
                    + torch.sign(Y_flat @ self.U_list[r])
                )
                Z_ens += self.gammas[r] * Z_r

            t_target += time.time() - tt0

            ATA += A_flat.T @ A_flat
            ATZ += A_flat.T @ Z_ens

        reg = self.lam * (N / 1000.0) * torch.eye(
            self.patch_dim, device=self.device
        )

        reg[-1, -1] = 1e-4

        self.W = torch.linalg.solve(ATA + reg, ATZ)

        return t_target, time.time() - t0

    @torch.no_grad()
    def forward(self, X, batch_size=1024):
        outputs = []

        for start in range(0, X.shape[0], batch_size):
            xb = X[start:min(start + batch_size, X.shape[0])]

            patches, H_out, W_out = self._extract_patches(xb)

            Z = patches @ self.W

            A = F.relu(Z).transpose(1, 2).reshape(
                xb.shape[0], self.out_channels, H_out, W_out
            )

            A = F.max_pool2d(A, kernel_size=2, stride=2)

            A = (
                A - A.mean(dim=(2, 3), keepdim=True)
            ) / (
                A.std(dim=(2, 3), keepdim=True) + 1e-5
            )

            outputs.append(A)

        return torch.cat(outputs, dim=0)

In [21]:
class FPEnsembleCNN:
    def __init__(
        self, num_classes, R=5,
        conv_channels=(64, 128, 256),
        hidden_mlp=1024,
        lam_conv=5.0, lam_fc=20.0,
        device='cpu'
    ):
        self.num_classes = num_classes
        self.R = R
        self.lam_fc = lam_fc
        self.device = device
        self.gammas = [1.0 / R] * R

        self.conv1 = Conv2DFPEnsembleBlock(
            1, conv_channels[0], 3, num_classes,
            R=R, lam=lam_conv, device=device
        )

        self.conv2 = Conv2DFPEnsembleBlock(
            conv_channels[0], conv_channels[1], 3, num_classes,
            R=R, lam=lam_conv, device=device
        )

        self.conv3 = Conv2DFPEnsembleBlock(
            conv_channels[1], conv_channels[2], 3, num_classes,
            R=R, lam=lam_conv, device=device
        )

        # After three pooling stages: 28 -> 14 -> 7 -> 3
        feat_dim = conv_channels[2] * (3 * 3 + 2) + 1

        self.hidden_mlp = hidden_mlp

        self.Q_fc = [
            torch.randn(feat_dim, hidden_mlp, device=device)
            / (feat_dim ** 0.5)
            for _ in range(R)
        ]

        self.U_fc = [
            torch.randn(num_classes, hidden_mlp, device=device)
            / (num_classes ** 0.5)
            for _ in range(R)
        ]

        self.W_fc = None
        self.W_out = None

    def _pool_features(self, C3):
        # Spatial flattening: 3 x 3
        flat = C3.reshape(C3.shape[0], -1)

        # Global average pooling
        g_avg = C3.mean(dim=(2, 3))

        # Global max pooling
        g_max = C3.amax(dim=(2, 3))

        feats = torch.cat([flat, g_avg, g_max], dim=1)

        feats = F.normalize(feats, p=2, dim=1)

        ones = torch.ones(
            feats.shape[0], 1, device=self.device
        )

        return torch.cat([feats, ones], dim=1)

    def fit(self, X, y):
        t_start = time.time()

        Y_onehot = F.one_hot(
            y, num_classes=self.num_classes
        ).float().to(self.device)

        total_target_time = 0.0

        # Conv Layer 1
        tt, _ = self.conv1.fit(
            X, Y_onehot, spatial_subsample=2
        )
        total_target_time += tt
        A1 = self.conv1.forward(X)

        # Conv Layer 2
        tt, _ = self.conv2.fit(
            A1, Y_onehot, spatial_subsample=1
        )
        total_target_time += tt
        A2 = self.conv2.forward(A1)

        # Conv Layer 3
        tt, _ = self.conv3.fit(
            A2, Y_onehot, spatial_subsample=1
        )
        total_target_time += tt
        A3 = self.conv3.forward(A2)

        # Dense FP-Ensemble hidden layer
        F_vec = self._pool_features(A3)

        tt0 = time.time()

        Z_fc_ens = torch.zeros(
            F_vec.shape[0], self.hidden_mlp,
            device=self.device
        )

        for r in range(self.R):
            Z_r = (
                torch.sign(F_vec @ self.Q_fc[r])
                + torch.sign(Y_onehot @ self.U_fc[r])
            )
            Z_fc_ens += self.gammas[r] * Z_r

        total_target_time += time.time() - tt0

        I_fc = self.lam_fc * torch.eye(
            F_vec.shape[1], device=self.device
        )
        I_fc[-1, -1] = 1e-4

        self.W_fc = torch.linalg.solve(
            F_vec.T @ F_vec + I_fc,
            F_vec.T @ Z_fc_ens
        )

        H_fc = F.relu(F_vec @ self.W_fc)

        H_fc = torch.cat(
            [H_fc, torch.ones(H_fc.shape[0], 1, device=self.device)],
            dim=1
        )

        # Final closed-form Ridge readout
        I_out = self.lam_fc * torch.eye(
            H_fc.shape[1], device=self.device
        )
        I_out[-1, -1] = 1e-4

        self.W_out = torch.linalg.solve(
            H_fc.T @ H_fc + I_out,
            H_fc.T @ Y_onehot
        )

        return total_target_time, time.time() - t_start

    @torch.no_grad()
    def predict(self, X):
        A1 = self.conv1.forward(X)
        A2 = self.conv2.forward(A1)
        A3 = self.conv3.forward(A2)

        F_vec = self._pool_features(A3)

        H_fc = F.relu(F_vec @ self.W_fc)

        H_fc = torch.cat(
            [H_fc, torch.ones(H_fc.shape[0], 1, device=self.device)],
            dim=1
        )

        return H_fc @ self.W_out

In [22]:
# Comparative evaluation: Original FP vs FP-Ensemble

ensemble_sizes = [1, 3, 5, 8]
seeds = [0, 1, 2]

results = []

for R in ensemble_sizes:

    train_accs = []
    test_accs = []
    test_aucs = []
    target_times = []
    total_times = []

    label_name = (
        "Original FP (R=1)"
        if R == 1
        else f"FP-Ensemble (R={R})"
    )

    print(f"\n--- {label_name} ---")

    for seed in seeds:

        torch.manual_seed(seed)
        np.random.seed(seed)

        if device == 'cuda':
            torch.cuda.manual_seed_all(seed)

        model = FPEnsembleCNN(
            num_classes=num_classes,
            R=R,
            conv_channels=(64, 128, 256),
            hidden_mlp=1024,
            lam_conv=5.0,
            lam_fc=20.0,
            device=device
        )

        t_target, t_total = model.fit(X_train, y_train_t)

        # Train accuracy
        train_logits = model.predict(X_train)

        train_acc = (
            train_logits.argmax(dim=1) == y_train_t
        ).float().mean().item() * 100

        # Test accuracy
        test_logits = model.predict(X_test)

        test_acc = (
            test_logits.argmax(dim=1) == y_test_t
        ).float().mean().item() * 100

        # Test AUC
        test_probs = F.softmax(
            test_logits, dim=1
        ).cpu().numpy()

        test_auc = roc_auc_score(
            y_test,
            test_probs,
            multi_class='ovr'
        ) * 100

        train_accs.append(train_acc)
        test_accs.append(test_acc)
        test_aucs.append(test_auc)
        target_times.append(t_target)
        total_times.append(t_total)

        print(
            f"Seed {seed} | "
            f"Train Acc: {train_acc:.2f}% | "
            f"Test Acc: {test_acc:.2f}% | "
            f"AUC: {test_auc:.2f}% | "
            f"Time: {t_total:.2f}s"
        )

        del model, train_logits, test_logits, test_probs

        if device == 'cuda':
            torch.cuda.empty_cache()

    results.append({
        "Architecture": label_name,
        "R": R,
        "Train Acc (%)": f"{np.mean(train_accs):.2f} ± {np.std(train_accs):.2f}",
        "Test Acc (%)": f"{np.mean(test_accs):.2f} ± {np.std(test_accs):.2f}",
        "Test AUC (%)": f"{np.mean(test_aucs):.2f} ± {np.std(test_aucs):.2f}",
        "Target Gen Time (s)": f"{np.mean(target_times):.2f}",
        "Total Train Time (s)": f"{np.mean(total_times):.2f}"
    })

# Final results
summary_df = pd.DataFrame(results)

print("\n=== Final Comparative Results ===")
print(summary_df.to_string(index=False))

display(summary_df)


--- Original FP (R=1) ---
Seed 0 | Train Acc: 100.00% | Test Acc: 94.52% | AUC: 99.76% | Time: 9.46s
Seed 1 | Train Acc: 100.00% | Test Acc: 95.27% | AUC: 99.65% | Time: 8.36s
Seed 2 | Train Acc: 100.00% | Test Acc: 95.23% | AUC: 99.59% | Time: 8.47s

--- FP-Ensemble (R=3) ---
Seed 0 | Train Acc: 100.00% | Test Acc: 93.68% | AUC: 99.83% | Time: 10.07s
Seed 1 | Train Acc: 100.00% | Test Acc: 95.73% | AUC: 99.85% | Time: 10.20s
Seed 2 | Train Acc: 100.00% | Test Acc: 95.06% | AUC: 99.81% | Time: 10.30s

--- FP-Ensemble (R=5) ---
Seed 0 | Train Acc: 100.00% | Test Acc: 96.54% | AUC: 99.68% | Time: 12.12s
Seed 1 | Train Acc: 100.00% | Test Acc: 95.64% | AUC: 99.85% | Time: 12.32s
Seed 2 | Train Acc: 100.00% | Test Acc: 96.77% | AUC: 99.83% | Time: 12.83s

--- FP-Ensemble (R=8) ---
Seed 0 | Train Acc: 100.00% | Test Acc: 93.14% | AUC: 99.82% | Time: 15.38s
Seed 1 | Train Acc: 100.00% | Test Acc: 95.71% | AUC: 99.83% | Time: 14.95s
Seed 2 | Train Acc: 100.00% | Test Acc: 95.27% | AUC: 99.64

,Architecture,R,Train Acc (%),Test Acc (%),Test AUC (%),Target Gen Time (s),Total Train Time (s)
0,Original FP (R=1),1,100.00 ± 0.00,95.01 ± 0.35,99.67 ± 0.07,0.13,8.76
1,FP-Ensemble (R=3),3,100.00 ± 0.00,94.83 ± 0.85,99.83 ± 0.02,0.07,10.19
2,FP-Ensemble (R=5),5,100.00 ± 0.00,96.31 ± 0.49,99.79 ± 0.08,0.10,12.42
3,FP-Ensemble (R=8),8,100.00 ± 0.00,94.71 ± 1.12,99.76 ± 0.09,0.20,15.05
